# Solving the Traveling Salesman Problem with Covalent and Fixstars Amplify

This example uses Covalent and Fixstars Amplify to solve the traveling salesman problem (TSP).

The code is based on a modified version of [Amplify Examples](https://github.com/fixstars/amplify-examples/blob/main/notebooks/ja/examples/tsp.ipynb). Amplify Examples is open source software under the [MIT License](https://github.com/fixstars/amplify-examples/blob/main/LICENSE).

## Environment Setup

### Steps to create a virtual environment on the remote machine.

To run this example, you need to setup an environment (or virtual environment) with the necessary packages installed on the remote machine.

Below are the steps to create a virtual environment on the remote machine named `amplify_env` with `venv`.

1. Login to the remote machine using your own account.
2. Verify that the Python version on the remote machine is the same as the one on your local machine, down to minor versions.
3. Create a virtual environment by running `python3 -m venv amplify_env`. The `amplify_env` directory is also created at this time.
4. Activate the virtual environment by running `source amplify_env/bin/activate`.
5. Install the necessary packages.
    * Please refer to [Dependent Packages](#dependencies) for the required packages.

<a id="dependencies"></a>
### Dependent Packages

This example requires the packages listed below. Please install the packages required to run this example in your local machine environment and in the virtual environment on the remote machine.  
Bolded packages are required for both the local machine and the virtual environment on the remote machine, and the rest are required only for the local machine.

This example uses covalent-pbspro-plugin.  
Please refer to the README of covalent-pbspro-plugin for information on how to install and use covalent-pbspro-plugin.

- **covalent**
- **amplify>=1.0.0**
- **numpy**
- matplotlib
- covalent-pbspro-plugin

In [ ]:
from __future__ import annotations
import covalent as ct

Create a PBSProExecutor object to execute tasks on the remote machine.

In [ ]:
remote_executor = ct.executor.PBSProExecutor(
    username="username",      # Enter your username on the remote machine.
    address="localhost",    # Enter the address of the remote machine.
    ssh_key_file="~/.ssh/id_rsa",  # Enter the path to your ssh key file.
    remote_workdir="$HOME/amplify_env",
    poll_freq=30,
    cleanup=True,
    embedded_qsub_args={
        "l": ["walltime=1:00:00"],
    },  # qsub options to be embedded in the script
    qsub_args={
    },  # qsub options to be given when it is run on the command line
    prerun_commands=[
        "source $HOME/amplify_env/bin/activate",
    ],
    postrun_commands=[],
    bashrc_path="~/.bashrc",
    log_stdout="stdout.log",
    log_stderr="stderr.log",
)

First, we prepare the functions needed to solve the traveling salesman problem.

The following is based on the code contained in [Amplify Examples](https://github.com/fixstars/amplify-examples/blob/main/notebooks/ja/examples/tsp.ipynb), and uses the covalent electron.

Note that the code below is implemented with Amplify SDK v1 and is not guaranteed to work with Amplify SDK v0.  
Please see following link for detail.
https://amplify.fixstars.com/en/docs/amplify/v1/migration.html

In [ ]:
import numpy as np

# This task is executed on the local machine.
@ct.electron
def gen_random_tsp(num_cities: int) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng()
    # coordinates
    locations = rng.random(size=(num_cities, 2))

    # distance matrix
    x = locations[:, 0]
    y = locations[:, 1]
    distances = np.sqrt(
        (x[:, np.newaxis] - x[np.newaxis, :]) ** 2
        + (y[:, np.newaxis] - y[np.newaxis, :]) ** 2
    )

    return locations, distances

In [ ]:
import amplify

# This task is executed on the remote machine.
@ct.electron(executor=remote_executor)
def solve_tsp(num_cities: int, distances: np.ndarray) -> np.ndarray:
    gen = amplify.VariableGenerator()
    q = gen.array("Binary", shape=(num_cities + 1, num_cities))
    q[num_cities, :] = q[0, :]

    objective: amplify.Poly = amplify.einsum("ij,ni,nj->", distances, q[:-1], q[1:])

    # Constraints on row direction
    row_constraints = amplify.one_hot(q[:-1], axis=1)

    # Constraints on column direction
    col_constraints = amplify.one_hot(q[:-1], axis=0)

    constraints = row_constraints + col_constraints

    constraints *= np.amax(distances)  # Set the weight of constraints
    model = objective + constraints

    client = amplify.FixstarsClient()
    client.token = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"  # Enter your token of Amplify AE.
    client.parameters.timeout = 5000  # timeout in milliseconds

    result = amplify.solve(model, client)

    if len(result) == 0:
        raise RuntimeError("Any one of constraints is not satisfied.")

    energy, values = result.best.objective, result.best.values
    q_values = q.evaluate(values)
    route = np.where(np.array(q_values) == 1)[1]

    return route

In [ ]:
import matplotlib.pyplot as plt

# Function to visualize the obtained TSP solution.
def show_route(route: list, distances: np.ndarray, locations: np.ndarray) -> np.float64:
    num_cities = len(route)
    path_length = sum(
        [distances[route[i]][route[(i + 1) % num_cities]] for i in range(num_cities)]
    )

    x = [i[0] for i in locations]
    y = [i[1] for i in locations]
    plt.figure(figsize=(7, 7))
    plt.title(f"path length: {path_length}")
    plt.xlabel("x")
    plt.ylabel("y")

    for i in range(num_cities):
        r = route[i]
        n = route[(i + 1) % num_cities]
        plt.plot([x[r], x[n]], [y[r], y[n]], "b-")
    plt.plot(x, y, "ro")
    plt.show()

    return path_length

Combine the above two `electron` to create a `lattice` workflow.

In [ ]:
@ct.lattice
def workflow(num_cities: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    locations, distances = gen_random_tsp(num_cities)
    route = solve_tsp(num_cities, distances)

    return route, distances, locations

Next, start the covalent server with the following command

```console
covalent start
````

With the default configuration, you can view the Covalent GUI by accessing http://localhost:48008/ from your browser.

Finally, execute the workflow. First, dispatch the workflow.

In [ ]:
dispatch_id = ct.dispatch(workflow)(10)

In [ ]:
print(dispatch_id)

In [ ]:
result = ct.get_result(dispatch_id, wait=True)
print(result)

Visualize the obtained solution.

In [ ]:
route, distances, locations = result.result

path_length = show_route(route, distances, locations)